# CUAD data preparation and audit

This notebook creates the frozen 20-class dataset used by the thesis experiments. It extracts CUAD clauses, performs document-title-group splitting, audits cross-split text reuse, trains the classical baseline, prepares PEFT files, and builds the training-only FAISS index.

The duplicate analysis is diagnostic. Legal agreements legitimately reuse boilerplate, so exact or near-exact matches are reported rather than automatically deleted. One placeholder-text label conflict (`[●]`) is isolated in the audit artifacts for transparency.

**Execution note:** running the saving cells writes to `/content/drive/MyDrive/Thesis1_Project` and replaces files with the same names. The retained outputs already document the authoritative completed run; rerun only when intentionally regenerating the data assets.

**Frozen output:** 8,257 training clauses, 833 validation clauses, 1,850 test clauses, and 20 classes.


## 1. Environment and project paths


In [ ]:
%pip install -q sentence-transformers faiss-cpu


In [ ]:
import ast
import json
import os
import pickle
import re
import zipfile

import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import NearestNeighbors

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DRIVE_ZIP_PATH = "/content/drive/MyDrive/Thesis1_Project/CUAD.zip"
COLAB_WORKSPACE = "/content/cuad_data"
SAVE_PATH = "/content/drive/MyDrive/Thesis1_Project"


## 2. Load CUAD

Mount Google Drive and extract the source archive only when the local Colab copy is absent.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Project paths

# Create working directories
os.makedirs(COLAB_WORKSPACE, exist_ok=True)
os.makedirs(SAVE_PATH, exist_ok=True)

# Extract the nested CUAD archives when needed.

inner_zip_path = os.path.join(COLAB_WORKSPACE, "CUAD_v1.zip")
final_data_file = os.path.join(COLAB_WORKSPACE, "CUAD_v1/CUAD_v1/master_clauses.csv")

if not os.path.exists(final_data_file):
    print("Extracting CUAD.zip...")
    with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(COLAB_WORKSPACE)

    if os.path.exists(inner_zip_path):
        print("Extracting inner CUAD_v1.zip...")
        with zipfile.ZipFile(inner_zip_path, 'r') as zip_ref:
            zip_ref.extractall(os.path.join(COLAB_WORKSPACE, "CUAD_v1"))
    print("Extraction complete.")
else:
    print("Data already extracted.")

# Verify extraction
if not os.path.exists(final_data_file):
    raise FileNotFoundError("Could not find master_clauses.csv. Check paths.")


## 3. Extract annotated clauses

Parse the 40 CUAD answer columns into one clause-level table. Parsing failures are recorded and stop execution rather than being silently discarded.


In [ ]:
master_df = pd.read_csv(final_data_file)
clause_columns = [column for column in master_df.columns if column.endswith("-Answer")]

rows = []
parse_failures = []
for row_index, row in master_df.iterrows():
    for clause_column in clause_columns:
        clause_type = clause_column.removesuffix("-Answer")
        raw_value = row[clause_type]
        if not isinstance(raw_value, str) or raw_value.strip() in {"[]", "", "nan"}:
            continue
        try:
            clause_texts = ast.literal_eval(raw_value)
        except (ValueError, SyntaxError, TypeError) as error:
            parse_failures.append({
                "row_index": int(row_index),
                "clause_type": clause_type,
                "error": str(error),
            })
            continue
        if not isinstance(clause_texts, list):
            parse_failures.append({
                "row_index": int(row_index),
                "clause_type": clause_type,
                "error": "Parsed value is not a list",
            })
            continue
        for clause_text in clause_texts:
            if isinstance(clause_text, str) and clause_text.strip():
                rows.append({
                    "contract_title": row["Document Name-Answer"],
                    "clause_text": clause_text.strip(),
                    "clause_type": clause_type,
                })

if parse_failures:
    failure_path = os.path.join(COLAB_WORKSPACE, "clause_parse_failures.csv")
    pd.DataFrame(parse_failures).to_csv(failure_path, index=False)
    raise ValueError(
        f"Clause extraction encountered {len(parse_failures)} parse failures. "
        f"Review {failure_path} before continuing."
    )

df_all_clauses = pd.DataFrame(rows)
df_all_clauses.to_csv(
    os.path.join(COLAB_WORKSPACE, "clauses_extracted_full.csv"),
    index=False,
)

print(f"Contracts loaded: {len(master_df):,}")
print(f"Annotated clause types: {len(clause_columns):,}")
print(
    f"Clauses extracted: {len(df_all_clauses):,} across "
    f"{df_all_clauses['clause_type'].nunique():,} types"
)


## 4. Freeze the classification task

Apply the predetermined 20-class label set and report the resulting imbalance. Class membership is not selected from the held-out test data.


In [ ]:
def class_imbalance_stats(series):
    """Compute entropy and imbalance ratio for class distribution."""
    counts = series.value_counts(normalize=True)
    entropy = -(counts * np.log2(counts)).sum()
    max_entropy = np.log2(len(counts))
    normalized_entropy = entropy / max_entropy if max_entropy > 0 else 0
    imbalance_ratio = counts.max() / counts.min() if counts.min() > 0 else np.inf
    return {
        "num_classes": len(counts),
        "entropy": entropy,
        "normalized_entropy": normalized_entropy,
        "imbalance_ratio": imbalance_ratio
    }

# Compute stats before filtering
print("Class Imbalance (Before Filtering):")
stats_before = class_imbalance_stats(df_all_clauses['clause_type'])
for k, v in stats_before.items():
    print(f"  {k:>20}: {v:.3f}" if isinstance(v, float) else f"  {k:>20}: {v}")

# IMPORTANT: Freeze the final 20 clause types.
# This avoids selecting classes dynamically from the full dataset, which would
# allow the held-out test set to influence the task definition.
FINAL_CLAUSE_TYPES = [
    "Parties",
    "License Grant",
    "Cap On Liability",
    "Audit Rights",
    "Anti-Assignment",
    "Insurance",
    "Document Name",
    "Agreement Date",
    "Governing Law",
    "Expiration Date",
    "Effective Date",
    "Post-Termination Services",
    "Revenue/Profit Sharing",
    "Minimum Commitment",
    "Exclusivity",
    "Rofr/Rofo/Rofn",
    "Ip Ownership Assignment",
    "Non-Transferable License",
    "Non-Compete",
    "Termination For Convenience",
]

TOP_N = len(FINAL_CLAUSE_TYPES)
missing_classes = sorted(set(FINAL_CLAUSE_TYPES) - set(df_all_clauses['clause_type'].unique()))
if missing_classes:
    raise ValueError(f"Required frozen clause types are missing from CUAD: {missing_classes}")

# Freeze class membership rather than recomputing the top-N classes from the full dataset.
df_filtered = df_all_clauses[
    df_all_clauses['clause_type'].isin(FINAL_CLAUSE_TYPES)
].reset_index(drop=True)

# Keep a deterministic class order for reports, labels, and downstream experiments.
df_filtered['clause_type'] = pd.Categorical(
    df_filtered['clause_type'],
    categories=FINAL_CLAUSE_TYPES,
    ordered=True
)

print(f"\nFiltered to the frozen set of {TOP_N} clause types")
print(f"Remaining: {len(df_filtered):,} clauses ({len(df_filtered)/len(df_all_clauses)*100:.1f}% of original)")
print("\nFrozen clause types:")
for i, label in enumerate(FINAL_CLAUSE_TYPES):
    print(f"  {i:02d}: {label}")


In [ ]:
# Compute stats after filtering
print("\nClass Imbalance (After Filtering):")
stats_after = class_imbalance_stats(df_filtered['clause_type'])
for k, v in stats_after.items():
    print(f"  {k:>20}: {v:.3f}" if isinstance(v, float) else f"  {k:>20}: {v}")

# Full class-frequency table (important because the classes are NOT perfectly balanced).
class_counts = (
    df_filtered['clause_type']
    .value_counts()
    .reindex(FINAL_CLAUSE_TYPES, fill_value=0)
    .rename_axis('clause_type')
    .reset_index(name='count')
)
class_counts['percentage'] = class_counts['count'] / len(df_filtered) * 100
print("\nClass distribution:")
display(class_counts)

# Visualize distribution
plt.figure(figsize=(12, 7))
plot_counts = class_counts.sort_values('count', ascending=True)
plt.barh(plot_counts['clause_type'], plot_counts['count'])
plt.title(f"Clause Type Distribution (Frozen {TOP_N} Classes)")
plt.xlabel("Count")
plt.ylabel("Clause Type")
plt.tight_layout()
plt.show()


## 5. Clean and describe clause text

Whitespace-only records are removed, while case and legal wording are preserved.


In [ ]:
# Preprocessing: strip whitespace, remove empty texts
# Note: Original case preserved for legal text (no lowercasing)
df_filtered['clause_text'] = df_filtered['clause_text'].astype(str).str.strip()
df_filtered = df_filtered[df_filtered['clause_text'] != ""].copy()

def analyze_clause_lengths(df, name="Dataset"):
    """Analyze and visualize clause text length distribution."""
    df['text_length'] = df['clause_text'].str.len()
    df['word_count'] = df['clause_text'].str.split().str.len()

    print(f"{name} Clause Length Statistics:")
    print(f"  Character length - Mean: {df['text_length'].mean():.0f}, Median: {df['text_length'].median():.0f}, Max: {df['text_length'].max()}")
    print(f"  Word count - Mean: {df['word_count'].mean():.0f}, Median: {df['word_count'].median():.0f}, Max: {df['word_count'].max()}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.hist(df['text_length'], bins=50, edgecolor='black', alpha=0.7)
    ax1.set_title(f'{name} - Character Length Distribution')
    ax1.set_xlabel('Characters')
    ax1.set_ylabel('Frequency')

    ax2.hist(df['word_count'], bins=50, edgecolor='black', alpha=0.7)
    ax2.set_title(f'{name} - Word Count Distribution')
    ax2.set_xlabel('Words')
    ax2.set_ylabel('Frequency')
    plt.tight_layout()
    plt.show()

    df.drop(['text_length', 'word_count'], axis=1, inplace=True)

analyze_clause_lengths(df_filtered, "Filtered Dataset")

print(f"Text cleaning complete. Total clauses: {len(df_filtered):,}")


## 6. Create document-title-group splits

Clauses sharing the same annotated document title are kept in one split. The grouping variable is derived from `Document Name-Answer`, preventing document-title-group overlap across training, validation, and test partitions. The resulting clause proportions are reported as observed rather than described as a nominal random ratio.


In [ ]:
# Split by annotated document-title group to reduce cross-split leakage.
# The two-stage GroupShuffleSplit produces approximately 68%/12%/20% of document-title groups
# before clause-count effects; actual clause proportions are reported below.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(df_filtered, groups=df_filtered["contract_title"]))

train_df = df_filtered.iloc[train_idx].reset_index(drop=True)
test_df = df_filtered.iloc[test_idx].reset_index(drop=True)

# Split validation from train
# This is intentionally performed at document-title-group level, not clause level.
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_SEED)
train_idx2, val_idx = next(gss_val.split(train_df, groups=train_df["contract_title"]))

train_df_final = train_df.iloc[train_idx2].reset_index(drop=True)
val_df = train_df.iloc[val_idx].reset_index(drop=True)

print("Dataset Split Summary:")
print(f"  Train:      {len(train_df_final):>6,} clauses")
print(f"  Validation: {len(val_df):>6,} clauses")
print(f"  Test:       {len(test_df):>6,} clauses")
print(f"  Total:      {len(df_filtered):>6,} clauses")

# Validate no data leakage between splits

def validate_split(train, val, test):
    """Ensure no data leakage between splits by checking document-title-group overlap."""
    train_contracts = set(train['contract_title'].unique())
    val_contracts = set(val['contract_title'].unique())
    test_contracts = set(test['contract_title'].unique())

    train_val_overlap = train_contracts & val_contracts
    train_test_overlap = train_contracts & test_contracts
    val_test_overlap = val_contracts & test_contracts

    if train_val_overlap:
        raise ValueError(f"Data leakage detected: {len(train_val_overlap)} document-title groups in both Train and Validation")
    if train_test_overlap:
        raise ValueError(f"Data leakage detected: {len(train_test_overlap)} document-title groups in both Train and Test")
    if val_test_overlap:
        raise ValueError(f"Data leakage detected: {len(val_test_overlap)} document-title groups in both Validation and Test")

    print("\nNo document-title-group overlap detected:")
    print(f"  Train groups: {len(train_contracts)}")
    print(f"  Validation groups: {len(val_contracts)}")
    print(f"  Test groups:  {len(test_contracts)}")
    return train_contracts, val_contracts, test_contracts

train_contracts, val_contracts, test_contracts = validate_split(
    train_df_final, val_df, test_df
)


In [ ]:
# Verify that every frozen class is represented in every split.
# If a class is absent from validation/test, downstream macro-F1 comparisons become
# harder to interpret, so stop rather than silently proceeding.
split_frames = {
    "train": train_df_final,
    "validation": val_df,
    "test": test_df,
}
split_counts = pd.DataFrame({
    name: frame['clause_type'].value_counts().reindex(FINAL_CLAUSE_TYPES, fill_value=0)
    for name, frame in split_frames.items()
})
split_counts['total'] = split_counts.sum(axis=1)

missing_by_split = {
    name: split_counts.index[split_counts[name] == 0].tolist()
    for name in split_frames
}
for name, missing in missing_by_split.items():
    if missing:
        raise ValueError(f"Frozen classes missing from {name} split: {missing}")

# Report exact clause counts and proportions (do not claim a nominal 70/15/15 split).
split_summary = pd.DataFrame({
    "clauses": [len(train_df_final), len(val_df), len(test_df)],
    "document_title_groups": [len(train_contracts), len(val_contracts), len(test_contracts)]
}, index=["train", "validation", "test"])
split_summary['clause_percentage'] = split_summary['clauses'] / len(df_filtered) * 100
split_summary['group_percentage'] = split_summary['document_title_groups'] / len(set(df_filtered['contract_title'])) * 100
print("\nActual split proportions:")
display(split_summary.round(2))

print("\nClass Distribution by Split:")
display(split_counts)

# Preserve the class label mapping for downstream experiments.
label2id = {label: i for i, label in enumerate(FINAL_CLAUSE_TYPES)}
id2label = {i: label for label, i in label2id.items()}


## 7. Audit exact and near-duplicate text

Normalization is used only for audit purposes. The approximate analysis reports directional cross-split nearest-neighbour records, not unique duplicate pairs.


In [ ]:
# Normalize text only for duplicate detection.
def normalize_for_duplicate(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

for frame in [train_df_final, val_df, test_df]:
    frame["normalized_clause_text"] = frame["clause_text"].map(normalize_for_duplicate)

train_texts = set(train_df_final["normalized_clause_text"])
val_texts = set(val_df["normalized_clause_text"])
test_texts = set(test_df["normalized_clause_text"])

exact_overlap = {
    "train_val": len(train_texts & val_texts),
    "train_test": len(train_texts & test_texts),
    "val_test": len(val_texts & test_texts),
}

print("Exact normalized-text overlap across splits:")
for pair, count in exact_overlap.items():
    print(f"  {pair}: {count}")

print("\nInterpretation: exact text overlap is reported for transparency; it is not automatically treated as leakage because legal contracts contain reusable boilerplate.")


In [ ]:
# Approximate template audit using character TF-IDF nearest neighbours.
combined_text = pd.concat([
    train_df_final["clause_text"], val_df["clause_text"], test_df["clause_text"]
], ignore_index=True)
combined_split = (
    ["train"] * len(train_df_final) +
    ["validation"] * len(val_df) +
    ["test"] * len(test_df)
)
all_labels = pd.concat([
    train_df_final["clause_type"], val_df["clause_type"], test_df["clause_type"]
], ignore_index=True)

char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_features=30000,
    sublinear_tf=True
)
X_char = char_vectorizer.fit_transform(combined_text)

nn = NearestNeighbors(n_neighbors=10, metric="cosine", n_jobs=-1)
distances, indices = nn.fit(X_char).kneighbors(X_char)

candidate_records = []
for i in range(len(combined_text)):
    # Find the nearest neighbour belonging to a different split.
    for rank in range(1, indices.shape[1]):
        j = int(indices[i, rank])
        if combined_split[i] != combined_split[j]:
            candidate_records.append({
                "similarity": 1.0 - float(distances[i, rank]),
                "index_a": i,
                "index_b": j,
                "split_a": combined_split[i],
                "split_b": combined_split[j],
                "class_a": str(all_labels.iloc[i]),
                "class_b": str(all_labels.iloc[j]),
                "text_a": str(combined_text.iloc[i]),
                "text_b": str(combined_text.iloc[j]),
            })
            break

candidate_records.sort(key=lambda x: x["similarity"], reverse=True)

print("\nTop 20 cross-split nearest-neighbour records:")
display(pd.DataFrame(candidate_records[:20])[[
    "similarity", "split_a", "split_b", "class_a", "class_b", "text_a", "text_b"
]])

high_similarity_count = sum(r["similarity"] >= 0.90 for r in candidate_records)
print(f"\nCross-split nearest-neighbour records with cosine similarity >= 0.90: {high_similarity_count}")
print("Interpretation: these records flag possible shared boilerplate/templates; they do not by themselves establish leakage.")


### 7.1 Detailed duplicate review and audit artifacts

Separate label-consistent boilerplate from exact text carrying conflicting labels, then save reproducible review tables.


In [ ]:
audit_frames = []
for split_name, frame in [("train", train_df_final), ("validation", val_df), ("test", test_df)]:
    temp = frame[["contract_title", "clause_type", "clause_text", "normalized_clause_text"]].copy()
    temp["split"] = split_name
    audit_frames.append(temp)

audit_all = pd.concat(audit_frames, ignore_index=True)

def unique_values(series):
    return sorted(set(str(x) for x in series))

grouped = audit_all.groupby("normalized_clause_text", dropna=False)
audit_table = grouped.agg(
    total_occurrences=("clause_text", "size"),
    train_count=("split", lambda s: int((s == "train").sum())),
    validation_count=("split", lambda s: int((s == "validation").sum())),
    test_count=("split", lambda s: int((s == "test").sum())),
    classes=("clause_type", unique_values),
    contracts=("contract_title", unique_values),
    example_text=("clause_text", "first"),
).reset_index()

audit_table["splits_present"] = audit_table.apply(
    lambda r: [
        name for name, count in [
            ("train", r.train_count),
            ("validation", r.validation_count),
            ("test", r.test_count)
        ] if count > 0
    ], axis=1
)
audit_table["num_splits"] = audit_table["splits_present"].str.len()
audit_table["label_consistent_all_splits"] = audit_table["classes"].map(lambda x: len(x) == 1)

exact_cross_split = audit_table[audit_table["num_splits"] >= 2].copy()
train_test_exact = audit_table[(audit_table["train_count"] > 0) & (audit_table["test_count"] > 0)].copy()

# IMPORTANT: determine train-test conflicts using ONLY train and test labels.
# A validation-only label must not make an otherwise consistent train/test duplicate a conflict.
def get_split_labels(normalized_text, split_name):
    return sorted(set(
        audit_all.loc[
            (audit_all["normalized_clause_text"] == normalized_text) &
            (audit_all["split"] == split_name),
            "clause_type"
        ].astype(str)
    ))

train_test_exact["train_labels"] = train_test_exact["normalized_clause_text"].map(
    lambda x: get_split_labels(x, "train")
)
train_test_exact["test_labels"] = train_test_exact["normalized_clause_text"].map(
    lambda x: get_split_labels(x, "test")
)
train_test_exact["train_test_label_consistent"] = train_test_exact.apply(
    lambda r: r["train_labels"] == r["test_labels"], axis=1
)

train_test_conflicts = train_test_exact[
    ~train_test_exact["train_test_label_consistent"]
].copy()

print("\nFINAL EXACT-DUPLICATE AUDIT")
print(f"Unique normalized texts appearing across >=2 splits: {len(exact_cross_split):,}")
print(f"Train-test exact normalized texts: {len(train_test_exact):,}")
print(f"Train-test exact texts with consistent train/test labels: {int(train_test_exact['train_test_label_consistent'].sum()):,}")
print(f"Train-test exact texts with conflicting train/test labels: {len(train_test_conflicts):,}")

if len(train_test_conflicts):
    print("\n*** TRAIN-TEST LABEL CONFLICTS — MANUAL REVIEW REQUIRED ***")
    display(train_test_conflicts[[
        "normalized_clause_text", "train_count", "test_count",
        "train_labels", "test_labels", "classes", "contracts", "example_text"
    ]])

    for _, conflict in train_test_conflicts.iterrows():
        text = conflict["normalized_clause_text"]
        print()
        print("CONFLICTING NORMALIZED TEXT:")
        print(repr(text))
        print("\nTRAIN OCCURRENCES:")
        display(audit_all[
            (audit_all["normalized_clause_text"] == text) &
            (audit_all["split"] == "train")
        ][["contract_title", "clause_type", "clause_text"]])
        print("TEST OCCURRENCES:")
        display(audit_all[
            (audit_all["normalized_clause_text"] == text) &
            (audit_all["split"] == "test")
        ][["contract_title", "clause_type", "clause_text"]])
else:
    print("\nNo train-test exact duplicate has conflicting labels.")

# Class summary for consistent train-test exact texts.
if len(train_test_exact):
    print("\nTrain-test exact duplicates by class:")
    class_summary = (
        train_test_exact[train_test_exact["train_test_label_consistent"]]
        .explode("classes")
        .groupby("classes")
        .size()
        .sort_values(ascending=False)
        .rename("exact_train_test_texts")
        .to_frame()
    )
    display(class_summary)


In [ ]:
# Near-duplicate summary: exclude exact matches (similarity ~1.0).
non_exact_high_similarity = [
    item for item in candidate_records
    if 0.90 <= item["similarity"] < 0.999999
]
print(f"\nCross-split nearest-neighbour records with 0.90 <= cosine similarity < 1.0: {len(non_exact_high_similarity):,}")

near_duplicate_review = pd.DataFrame(non_exact_high_similarity[:50])
if not near_duplicate_review.empty:
    near_duplicate_review = near_duplicate_review[[
        "similarity", "split_a", "split_b", "class_a", "class_b", "text_a", "text_b"
    ]].copy()
    near_duplicate_review["similarity"] = near_duplicate_review["similarity"].round(4)
    display(near_duplicate_review)
else:
    print("No non-exact cross-split nearest-neighbour records at cosine similarity >= 0.90 were found.")

# Save reproducible audit artifacts.
exact_audit_path = os.path.join(SAVE_PATH, "exact_duplicate_audit.csv")
train_test_audit_path = os.path.join(SAVE_PATH, "train_test_exact_duplicate_audit.csv")
train_test_conflict_path = os.path.join(SAVE_PATH, "train_test_label_conflicts.csv")
near_audit_path = os.path.join(SAVE_PATH, "near_duplicate_audit.csv")

exact_cross_split.drop(columns=["splits_present"], errors="ignore").to_csv(exact_audit_path, index=False)
train_test_exact.drop(columns=["splits_present"], errors="ignore").to_csv(train_test_audit_path, index=False)
train_test_conflicts.drop(columns=["splits_present"], errors="ignore").to_csv(train_test_conflict_path, index=False)
near_duplicate_review.to_csv(near_audit_path, index=False)

audit_summary = {
    "exact_overlap": exact_overlap,
    "cross_split_unique_exact_texts": int(len(exact_cross_split)),
    "train_test_exact_texts": int(len(train_test_exact)),
    "train_test_exact_label_consistent": int(train_test_exact["train_test_label_consistent"].sum()) if len(train_test_exact) else 0,
    "train_test_exact_label_conflicts": int(len(train_test_conflicts)),
    "near_duplicate_records_0_90_to_below_1": int(len(non_exact_high_similarity)),
    "policy": "diagnostic_only_no_records_removed_automatically",
    "conflict_file": "train_test_label_conflicts.csv",
}
with open(os.path.join(SAVE_PATH, "duplicate_audit_summary.json"), "w") as f:
    json.dump(audit_summary, f, indent=2)

print("\nAudit files saved:")
for p in [exact_audit_path, train_test_audit_path, train_test_conflict_path, near_audit_path]:
    print(f"  - {p}")
print(f"  - {os.path.join(SAVE_PATH, 'duplicate_audit_summary.json')}")

# Remove audit-only columns before downstream modeling/saving.
for frame in [train_df_final, val_df, test_df]:
    frame.drop(columns=["normalized_clause_text"], inplace=True, errors="ignore")


## 8. Train the classical baseline

Fit TF-IDF unigrams and bigrams with balanced logistic regression on the frozen training split, then evaluate validation and test performance.


In [ ]:
print("Training TF-IDF + Logistic Regression baseline...")

# Prepare TF-IDF features
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train = tfidf.fit_transform(train_df_final['clause_text'])
X_val = tfidf.transform(val_df['clause_text'])
X_test = tfidf.transform(test_df['clause_text'])

y_train = train_df_final['clause_type']
y_val = val_df['clause_type']
y_test = test_df['clause_type']

# Train model
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_SEED)
lr.fit(X_train, y_train)


In [ ]:
# Evaluate
y_train_pred = lr.predict(X_train)
y_val_pred = lr.predict(X_val)
y_test_pred = lr.predict(X_test)

print("\nBaseline Results:")
print("\n--- Validation Set ---")
print(classification_report(y_val, y_val_pred, zero_division=0))

print("\n--- Test Set ---")
print(classification_report(y_test, y_test_pred, zero_division=0))

# Save baseline results for comparison
baseline_results = {
    "model": "TF-IDF + Logistic Regression",
    "preprocessing": "unigrams + bigrams, max_features=5000",
    "train_accuracy": float(accuracy_score(y_train, y_train_pred)),
    "val_accuracy": float(accuracy_score(y_val, y_val_pred)),
    "test_accuracy": float(accuracy_score(y_test, y_test_pred)),
    "val_macro_f1": float(f1_score(y_val, y_val_pred, average='macro', zero_division=0)),
    "test_macro_f1": float(f1_score(y_test, y_test_pred, average='macro', zero_division=0)),
    "val_weighted_f1": float(f1_score(y_val, y_val_pred, average='weighted', zero_division=0)),
    "test_weighted_f1": float(f1_score(y_test, y_test_pred, average='weighted', zero_division=0))
}

print("\nBaseline Summary:")
for k, v in baseline_results.items():
    if isinstance(v, float):
        print(f"  {k:>20}: {v:.4f}")
    else:
        print(f"  {k:>20}: {v}")

# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred, labels=lr.classes_)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=lr.classes_, yticklabels=lr.classes_, cmap='Blues')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.title("Baseline Confusion Matrix - Test Set")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


## 9. Prepare PEFT records

Write the frozen training and validation examples in instruction/input/output JSONL format.


In [ ]:
def df_to_jsonl(df, file_path):
    """Convert DataFrame to instruction-tuned JSONL format."""
    with open(file_path, 'w', encoding='utf-8') as f:
        for _, row in df.iterrows():
            entry = {
                "instruction": "Classify the following contract clause into its correct clause type:",
                "input": row['clause_text'],
                "output": row['clause_type']
            }
            f.write(json.dumps(entry) + "\n")
    print(f"Saved {len(df)} entries to {file_path}")

# Create JSONL files
df_to_jsonl(train_df_final, os.path.join(SAVE_PATH, "peft_train.jsonl"))
df_to_jsonl(val_df, os.path.join(SAVE_PATH, "peft_val.jsonl"))


## 10. Build the training-only retrieval index

Encode only training clauses with `all-MiniLM-L6-v2` and store raw 384-dimensional embeddings in `IndexFlatL2`. Query embeddings must therefore remain unnormalized.


In [ ]:
# RAG Configuration:
# - Index Type: FAISS IndexFlatL2 (L2 distance)
# - Normalization: FALSE (raw embeddings)
# - Embedding Model: all-MiniLM-L6-v2
# Note: Query normalization must match this setting in experiments

# Prepare training clauses
train_clauses = train_df_final[['clause_text', 'clause_type']].reset_index(drop=True).copy()
print(f"Preparing {len(train_clauses):,} training clauses for RAG index...")

# Load embedding model
print("Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings
print("Encoding clauses (this may take 2-3 minutes)...")
embeddings = embedding_model.encode(
    train_clauses['clause_text'].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
    batch_size=32
)

assert len(train_clauses) == len(embeddings), "Mismatch between clauses and embeddings!"
print(f"Generated {embeddings.shape[0]:,} embeddings of dimension {embeddings.shape[1]}")


In [ ]:
# Build FAISS index (no normalization - using L2 distance)
d = embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(embeddings.astype('float32'))

print(f"FAISS index built successfully ({index.ntotal:,} vectors)")

# Prepare metadata
rag_metadata = train_clauses.to_dict(orient='records')


if index.ntotal != len(rag_metadata) or index.ntotal != len(train_df_final):
    raise RuntimeError("RAG index, metadata and training rows are misaligned.")


## 11. Save authoritative artifacts

Save the frozen splits, baseline metrics, retrieval assets, duplicate audits, and the configuration consumed by the experiment notebook.


In [ ]:
train_df_final.to_csv(os.path.join(SAVE_PATH, "train.csv"), index=False)
val_df.to_csv(os.path.join(SAVE_PATH, "val.csv"), index=False)
test_df.to_csv(os.path.join(SAVE_PATH, "test.csv"), index=False)

faiss.write_index(index, os.path.join(SAVE_PATH, "rag_index.faiss"))
with open(os.path.join(SAVE_PATH, "rag_metadata.pkl"), "wb") as file:
    pickle.dump(rag_metadata, file)

with open(os.path.join(SAVE_PATH, "baseline_results.json"), "w") as file:
    json.dump(baseline_results, file, indent=2)

print("Saved the frozen CSV splits, RAG assets and baseline metrics.")


In [ ]:
# Save comprehensive configuration file
config = {
    "dataset_info": {
        "name": "CUAD (Contract Understanding Atticus Dataset)",
        "source": "https://www.atticusprojectai.org/cuad",
        "original_contracts": int(master_df.shape[0]),
        "original_clause_types": int(df_all_clauses['clause_type'].nunique()),
        "original_clauses": int(len(df_all_clauses)),
        "final_num_classes": int(len(FINAL_CLAUSE_TYPES)),
        "final_clause_types": FINAL_CLAUSE_TYPES,
        "class_selection": "Frozen 20-class list; class membership is not recomputed from held-out data",
        "random_seed": RANDOM_SEED
    },
    "splits": {
        "train": int(len(train_df_final)),
        "validation": int(len(val_df)),
        "test": int(len(test_df)),
        "total": int(len(df_filtered)),
        "train_document_title_groups": int(len(train_contracts)),
        "validation_document_title_groups": int(len(val_contracts)),
        "test_document_title_groups": int(len(test_contracts)),
        "train_clause_percentage": float(len(train_df_final) / len(df_filtered) * 100),
        "validation_clause_percentage": float(len(val_df) / len(df_filtered) * 100),
        "test_clause_percentage": float(len(test_df) / len(df_filtered) * 100),
        "strategy": "Two-stage GroupShuffleSplit by annotated document-title group"
    },
    "label_mapping": {
        "label2id": label2id,
        "id2label": {str(k): v for k, v in id2label.items()}
    },
    "class_distribution": {
        "overall": {str(k): int(v) for k, v in class_counts.set_index('clause_type')['count'].items()},
        "train": {str(k): int(v) for k, v in split_counts['train'].items()},
        "validation": {str(k): int(v) for k, v in split_counts['validation'].items()},
        "test": {str(k): int(v) for k, v in split_counts['test'].items()}
    },
    "class_imbalance": {
        "before_filtering": stats_before,
        "after_filtering": stats_after,
        "primary_metric": "macro_f1"
    },
    "duplicate_audit": {
        "exact_overlap": exact_overlap,
        "train_test_exact_texts": int(len(train_test_exact)),
        "train_test_label_conflicts": int(len(train_test_conflicts)),
        "detailed_review": "See exact_duplicate_audit.csv, train_test_exact_duplicate_audit.csv, train_test_label_conflicts.csv, near_duplicate_audit.csv, and duplicate_audit_summary.json",
        "policy": "Diagnostic only; no records removed automatically"
    },
    "preprocessing": {
        "lowercased": False,
        "stripped_whitespace": True,
        "removed_empty": True,
        "description": "Original case preserved for legal text"
    },
    "models": {
        "base_llm": "mistralai/Mistral-7B-Instruct-v0.2",
        "embedding_model": "all-MiniLM-L6-v2"
    },
    "rag_config": {
        "index_type": "IndexFlatL2",
        "normalized_embeddings": False,
        "embedding_dim": int(d),
        "num_vectors": int(index.ntotal),
        "distance_metric": "L2 (Euclidean)",
        "important_note": "Queries MUST NOT be normalized when using this index!"
    },
    "baseline_performance": baseline_results,
    "split_strategy": "GroupShuffleSplit using Document Name-Answer as the grouping variable"
}

with open(os.path.join(SAVE_PATH, "dataset_config.json"), "w") as f:
    json.dump(config, f, indent=2)


## 12. Final integrity check

Confirm that every required downstream artifact exists and report the authoritative split and baseline summary.


In [ ]:
required_artifacts = [
    "train.csv",
    "val.csv",
    "test.csv",
    "peft_train.jsonl",
    "peft_val.jsonl",
    "rag_index.faiss",
    "rag_metadata.pkl",
    "baseline_results.json",
    "dataset_config.json",
    "exact_duplicate_audit.csv",
    "train_test_exact_duplicate_audit.csv",
    "train_test_label_conflicts.csv",
    "near_duplicate_audit.csv",
    "duplicate_audit_summary.json",
]
missing_artifacts = [
    name for name in required_artifacts
    if not os.path.exists(os.path.join(SAVE_PATH, name))
]
if missing_artifacts:
    raise FileNotFoundError(f"Missing final artifacts: {missing_artifacts}")

final_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "clauses": [len(train_df_final), len(val_df), len(test_df)],
    "contracts": [len(train_contracts), len(val_contracts), len(test_contracts)],
    "clause_percentage": [
        len(train_df_final) / len(df_filtered) * 100,
        len(val_df) / len(df_filtered) * 100,
        len(test_df) / len(df_filtered) * 100,
    ],
})
display(final_summary.round(2))
print(f"Artifacts verified: {len(required_artifacts)}")
print(f"Baseline test accuracy: {baseline_results['test_accuracy']:.4f}")
print(f"Baseline test Macro F1: {baseline_results['test_macro_f1']:.4f}")
print(f"Baseline test Weighted F1: {baseline_results['test_weighted_f1']:.4f}")
